<a href="https://colab.research.google.com/github/tardigrade-dot/colab-script/blob/main/03_generate_srt_funasr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

!pip install -r https://raw.githubusercontent.com/FunAudioLLM/Fun-ASR/refs/heads/main/requirements.txt
!pip install wetext


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    # print(f"PyTorch CUDA version: {torch.backends.cuda.version()}")

# Uninstall and reinstall torch, torchvision, and torchaudio to ensure CUDA compatibility
# This often fixes CUDA version mismatches in Colab environments.
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!git clone https://github.com/FunAudioLLM/Fun-ASR

In [ ]:
import os
os.chdir('/content/Fun-ASR')
current_directory = os.getcwd()
print(f"Current working directory: {current_directory}")


In [ ]:
!pip install huggingface_hub

In [ ]:
!hf download FunAudioLLM/Fun-ASR-Nano-2512 --local-dir /content/Fun-ASR-Nano-2512

In [ ]:
!hf download FunAudioLLM/SenseVoiceSmall --local-dir /content/SenseVoiceSmall

In [ ]:
import sys
import os

current_dir = os.getcwd()
directory_to_add = '/content/Fun-ASR'
if directory_to_add not in sys.path:
    sys.path.insert(0, directory_to_add) # insert at the beginning to give it priority
    print(f"Added '{directory_to_add}' to sys.path")


In [ ]:
import re
import os
from pathlib import Path
import string
from funasr import AutoModel
import difflib
from pydub import AudioSegment
from wetext import Normalizer

class CtcForcdAlign:
    def __init__(self, model_dir, device) -> None:

        self.MIN_CHARS_PER_LINE = 10
        self.model = AutoModel(
            model=model_dir,
            vad_model="fsmn-vad",
            vad_kwargs={"max_single_segment_time": 30_000},
            device=device,
            disable_update=True,
            trust_remote_code=True,
            # remote_code="./model.py",
            remote_code="funasr.models.sense_voice.model",
        )

        self.normalizer = Normalizer(lang="zh", operator="tn", remove_erhua=True, traditional_to_simple=True)
        chinese_pattern = r"（.*?）"
        english_pattern = r"\([^)]*?\)"

        self.combined_pattern = f"{chinese_pattern}|{english_pattern}"
        self.PUNCTUATION_CHARS = set(string.punctuation) | set(',.?，。！？；：、“”‘’《》【】（）')
        self.punctuation_regex = r'[.,:;!?，。、；：？！“‘”’·（）「」《》【】\s-]'
        self.PUNCT_REGEX = re.compile(self.punctuation_regex)

    def is_punctuation(self, token):
        return token in self.PUNCTUATION_CHARS

    def map_opcodes_to_raw(self, clean_opcodes, asr_timestamps, correct_tokens_raw):
        aligned_timestamps = []
        status = []

        j_raw_ptr = 0

        for tag, i1, i2, j1, j2 in clean_opcodes:

            while j_raw_ptr < len(correct_tokens_raw) and self.PUNCT_REGEX.match(correct_tokens_raw[j_raw_ptr]):
                aligned_timestamps.append((None, None))
                j_raw_ptr += 1

            asr_chars_consumed = i2 - i1
            if tag == "equal":
                for k in range(j2 - j1): # 遍历匹配的汉字
                    aligned_timestamps.append(asr_timestamps[i1 + k])
                    status.append(True)
                    j_raw_ptr += 1 # 原始指针移动一个汉字
                    while j_raw_ptr < len(correct_tokens_raw) and self.PUNCT_REGEX.match(correct_tokens_raw[j_raw_ptr]):
                        aligned_timestamps.append((None, None))
                        j_raw_ptr += 1
            elif tag == "replace" or tag == "insert":
                for k in range(j2 - j1): # 遍历差异的汉字
                    if asr_chars_consumed > 0:
                        idx = i1 + k % asr_chars_consumed
                        aligned_timestamps.append(asr_timestamps[idx])
                    else: # 纯粹的插入 (tag='insert')
                        aligned_timestamps.append((None, None))
                    status.append(False)
                    j_raw_ptr += 1 # 原始指针移动一个汉字
                    while j_raw_ptr < len(correct_tokens_raw) and self.PUNCT_REGEX.match(correct_tokens_raw[j_raw_ptr]):
                        aligned_timestamps.append((None, None))
                        j_raw_ptr += 1
            elif tag == "delete":
                continue

        while j_raw_ptr < len(correct_tokens_raw) and self.PUNCT_REGEX.match(correct_tokens_raw[j_raw_ptr]):
            aligned_timestamps.append((None, None))
            j_raw_ptr += 1
        if len(aligned_timestamps) == len(correct_tokens_raw):
            return aligned_timestamps, status
        else:
            print(f"🚨 警告: 最终长度不匹配. Raw: {len(correct_tokens_raw)}, Aligned: {len(aligned_timestamps)}")
            return aligned_timestamps, status

    def map_asr_to_correct(self, asr_tokens, asr_timestamps, correct_tokens):

        clean_correct_tokens = re.sub(self.punctuation_regex, '', correct_tokens)
        matcher_clean = difflib.SequenceMatcher(None, asr_tokens, clean_correct_tokens)
        clean_opcodes = matcher_clean.get_opcodes()

        return self.map_opcodes_to_raw(clean_opcodes, asr_timestamps, correct_tokens)

    def is_valid_time(self, t):
        return isinstance(t, (int, float)) and t >= 0

    def get_valid_start_end(self, words):
        start_time, end_time = None, None

        for w in words:
            t = w.get('start')
            if self.is_valid_time(t):
                start_time = t
                break

        for w in reversed(words):
            t = w.get('end')
            if self.is_valid_time(t):
                end_time = t
                break

        return start_time, end_time

    def ms_to_srt_time_format(self, ms):
        # 确保输入是整数或可以转换为整数
        ms = int(ms)
        # 计算小时、分钟、秒和毫秒
        hours = ms // 3600000
        minutes = (ms % 3600000) // 60000
        seconds = (ms % 60000) // 1000
        milliseconds = ms % 1000

        # 使用 f-string 格式化输出，确保每个部分都有固定的位数
        return f"{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}"

    def generate_srt_from_words_and_timestamps(self, words, timestamps):
        if not words or not timestamps or len(words) != len(timestamps):
            print(f"words length[{len(words)}] not equal to timestamps length[{len(timestamps)}]")
            raise Exception('process error')

        srt_content = ""
        subtitle_index = 1
        sentence_delimiters = ['。', '？', '！', '.', '?', '!', '…']

        current_subtitle_words = []
        end_time = 0

        # 内部函数：负责判断是否满足长度要求并输出字幕
        def _write_subtitle(words_to_write, pre_end_time):
            nonlocal srt_content, subtitle_index

            if not words_to_write:
                return None

            # start_time = words_to_write[0]['start']
            # end_time = words_to_write[-1]['end']
            try:

                start_time, end_time = self.get_valid_start_end(words_to_write)
                if pre_end_time and pre_end_time != 0 and start_time < pre_end_time:
                    start_time = pre_end_time
            except Exception as e:
                raise Exception(f"获取有效时间戳时出错, 音频质量存在问题: {e}")

            # 将单词列表组合成一个完整的句子
            sentence_text = "".join([w['word'] for w in words_to_write])

            srt_content += str(subtitle_index) + "\n"
            srt_content += f"{self.ms_to_srt_time_format(start_time)} --> {self.ms_to_srt_time_format(end_time)}\n"
            srt_content += sentence_text + "\n\n"
            subtitle_index += 1
            return end_time

        for i in range(len(words)):
            word = words[i]
            timestamp = timestamps[i]

            current_subtitle_words.append({'word': word, 'start': timestamp[0], 'end': timestamp[1]})

            is_sentence_end = word in sentence_delimiters
            if is_sentence_end:
                current_text = "".join([w['word'] for w in current_subtitle_words])
                current_length = len(current_text)
                if current_length >= self.MIN_CHARS_PER_LINE:
                    end_time = _write_subtitle(current_subtitle_words, end_time)
                    current_subtitle_words = [] # 清空累计列表
        if current_subtitle_words:
            all_none = all(
                item['start'] is None and item['end'] is None
                for item in current_subtitle_words
            )
            if not all_none:
                _write_subtitle(current_subtitle_words, end_time)

        return srt_content

    def generate_srt_file(self, wav_file, over_write=False):

        print(f'generate srt file for wav {wav_file}')
        wav_path = Path(wav_file)

        txt_path = wav_path.with_suffix('.txt')
        srt_path = wav_path.with_suffix('.srt')

        txt_file = str(txt_path)
        srt_file = str(srt_path)

        print(f'\nwav [{wav_file}], \ntxt [{txt_file}], \nsrt [{srt_file}]')

        if not wav_path.exists():
            raise Exception(f'wav file[{wav_file}] not exists ')
        if not txt_path.exists():
            raise Exception(f'txt file[{txt_file}] not exists ')
        if srt_path.exists() and not over_write:
            print(f'srt file[{srt_file}] exists, just return ')
            return

        with open(txt_file, 'r') as f:
            target_txt = "".join(f.readlines())

        res = self.model.generate(
                input=wav_file,
                cache={},
                language="auto",  # "zn", "en", "yue", "ja", "ko", "nospeech"
                use_itn=False,
                batch_size_s=30,
                merge_vad=True,
                merge_length_s=20,
                output_timestamp=True,
                # batch_size=1
            )
        output = res[0]
        print("output is : ", "".join(output["words"]))
        target_txt = re.sub(self.combined_pattern, "", target_txt)
        target_txt = target_txt.replace("\n", " ")
        target_txt = self.normalizer.normalize(target_txt)

        aligned_timestamps, status = self.map_asr_to_correct(output["words"], output["timestamp"], target_txt)

        num_tokens = len(status)
        num_mismatch = status.count(False)
        mismatch_ratio = num_mismatch / num_tokens

        if mismatch_ratio > 0.05:
            print(f"⚠️ ❌ 注意, 不匹配率过高.  不匹配 token({num_mismatch}/{num_tokens}) 占比: {mismatch_ratio:.2%}")
        else:
            print(f"⚠️ ✅ 不匹配 token({num_mismatch}/{num_tokens}) 占比: {mismatch_ratio:.2%}")

        srt_result = self.generate_srt_from_words_and_timestamps(target_txt, aligned_timestamps)

        with open(srt_file, 'w') as f:
            f.write(srt_result)

        print(f"字幕已生成, 保存在:{srt_file}")

    def get_wav_list_sorted(self, wav_src_dir, wav_regex=None):

        if wav_regex is None:
            wav_regex = r'.*-(\d+)_(\d+)\.wav$'
        pattern = re.compile(wav_regex)

        files = [
            f for f in os.listdir(wav_src_dir)
            if f.endswith('.wav') and pattern.match(f)
        ]

        def sort_key(filename):
            m = pattern.match(filename)
            if m:
                second, first = map(int, m.groups())
                return (second, first)
            return (float('inf'), float('inf'))

        sorted_files = sorted(files, key=sort_key)

        sorted_paths = [os.path.join(wav_src_dir, f) for f in sorted_files]
        return sorted_paths

    def check_srt_exsis(self, wav_src_dir):
        srt_result = []
        for w_path in self.get_wav_list_sorted(wav_src_dir):

            srt_path = Path(w_path).with_suffix('.srt')

            if not srt_path.exists():
                srt_result.append(srt_path.stem)

        print(f'以下文件没有srt文件, 请检查语音文件 {srt_result}')

    def generate_srt_dir(self, wav_src_dir, over_write=False):
        for w_path in self.get_wav_list_sorted(wav_src_dir):

            try:
                self.generate_srt_file(w_path, over_write)
            except Exception as e:
                print(f"处理文件 {w_path} 时发生错误：{e}")

book_name = "wenhuaquanliyuguojia"
cfa = CtcForcdAlign(
    # model_dir = "/Volumes/sw/pretrained_models/SenseVoiceSmall",
    # model_dir = "/content/Fun-ASR-Nano-2512",
    model_dir = "/content/SenseVoiceSmall",
    device = "cpu"
)

cfa.generate_srt_dir(f"/content/drive/MyDrive/{book_name}")

# cfa.generate_srt_dir(f"/Volumes/sw/tts_result/{book_name}", over_write=False)
cfa.check_srt_exsis(f"/Volumes/sw/tts_result/{book_name}")

# cfa.generate_srt_file("/Volumes/sw/tts_result/sulianjianshi/sulianjianshi-4_2.wav", True)
# cfa.generate_srt_dir('/Volumes/sw/MyDrive/zhengzhi1/output', r"zhengzhi1-(\d+)_(\d+)\.wav", True)





100%|██████████| 1/1 [00:08<00:00,  8.95s/it]


{'load_data': '0.000', 'extract_feat': '0.063', 'forward': '8.953', 'batch_size': '1', 'rtf': '0.298'}, : 100%|██████████| 1/1 [00:08<00:00,  8.95s/it]


rtf_avg: 0.298: 100%|██████████| 1/1 [00:08<00:00,  8.96s/it]



  0%|          | 0/1 [00:00<?, ?it/s]


100%|██████████| 1/1 [00:06<00:00,  6.92s/it]


{'load_data': '0.000', 'extract_feat': '0.053', 'forward': '6.921', 'batch_size': '1', 'rtf': '0.231'}, : 100%|██████████| 1/1 [00:06<00:00,  6.92s/it]


rtf_avg: 0.231: 100%|██████████| 1/1 [00:06<00:00,  6.94s/it]


100%|██████████| 1/1 [02:22<00:00, 142.11s/it]

rtf_avg: 0.277, time_speech:  511.920, time_escape: 141.599: 100%|██████████| 1/1 [02:22<00:00, 142.12s/it]


output is :  与乡村集团相关的另一类公务多兴起于一九零琅到一九一二年这包括看亲管理村学和村庄自卫等等一般情况下村庄领袖们还得掌握村庄收集和账务对此他们是向外保密的一九三零年之后河北省村长会电不另零览控制的乡村政治这从会头们的经济状况可以得到一些反应而且精英们还把持着村长副的职位民国之前会头中吴会长一切决定由集体做出一九一一年之后根据政府统一北方之后推行的乡镇和吕林制是集行政和自卫于一身的组织各级长所层层选出最下一层为五家组成的林林社长林长也是选举旅长的代表但实际上在三十年代只有个别村庄实行了选举制而吕林制与过去十家连作的保假制亦无大的区别三十年代中乡的规模和职能经常变动直到一九四一年以后在日伪政权强力推行下大乡制才基本取代了自然村落制村庄领袖群像下面将结合财产占有和传记文字等资料来分析地产和非地产宗族保护关系等因素是如何决定乡村领导层的构成的我还将探讨进入二十世纪之后这些因素在乡村政治中作用减弱的程度沙井村满铁调查人员认为沙井是华北平原一个比较典型的村庄到二十世纪三十年代末该村有七十户人家村中没有拥有许多店户的大出租地主但有数家拥有土地七十亩以上雇佣长工或短工的经营型地主土地分配很不平均占全村总户数百分之六十的农户共占全村百分之十四的土地而占总户数百分之十五的富户却占全村土地的百分之二百沙井村的领导权掌握在占总户数百分之十五的富有人家手中而且不同于其他满铁调查过的村庄这种富人掌权的状况一直持续到二十世纪三十年代一九零零年根据上级命令正式成立了青苗会会首们的主要职责从宗教方面转向沪丘等市在此之前沪丘是各自进行的青苗会和会手制不仅是一个保护庄稼的村庄组织而且国家政权可以借此有效的征收捐税和贪款村工会的性质如何据二十世纪三十年代及更早时期村工会成员占有土地的零星资料表明拥有财富是进入乡村领导层的关键这在早期更是如此沙井村九个会首平均占有土地五十亩远远高于全村户均土地面积数从历史资料来看二十世纪初期的村庄首饰比四十年代的首饰更为富有在一九四零年的九位首饰中有七位其上辈也任过首饰而且那时更为富有另一些村民其先辈富有时曾认过首饰但由于家道中落便自然而然被淘汰出首饰行列不过在一九四零年重新实行保假制使两个并不富有的人被任命为假长这表明到明国晚期村中财主开始躲避村中行政职务对历任村长姓名稍作分析便发现从清末开始李氏一族基本上控制着这一职位清末时李真任村长后来其子李振英其



  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:06<00:00,  6.32s/it]

{'load_data': '2.258', 'extract_feat': '3.772', 'forward': '6.321', 'batch_size': '1', 'rtf': '0.114'}, : 100%|██████████| 1/1 [00:06<00:00,  6.32s/it]

rtf_avg: 0.114: 100%|██████████| 1/1 [00:06<00:00,  6.34s/it]


  0%|          | 0/1 [00:00<?, ?it/s]


  0%|          | 0/1 [00:00<?, ?it/s]


100%|██████████| 1/1 [00:00<00:00,  1.75it/s]


{'load_data': '0.000', 'extract_feat': '0.010', 'forward': '0.571', 'batch_size': '1', 'rtf': '1.057'}, : 100%|██████████| 1/1 [00:00<00:00,  1.75it/s]


rtf_avg: 1.057: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s]



  0%|          | 0/1 [00:00<?, ?it/s]


100%|██████████| 1/1 [00:00<00:00,  1.78it/s]


{'load_data': '0.000', 'extract_feat': '0.007', 'forward': '0.561', 'batch_size': '1', 'rtf': '0.935'}, : 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]


rtf_avg: 0.935: 100%|██████████| 1/1 [00:00<00:00,  1.73it/s]



  0%|          | 0/1 [00:00<?, ?it/s]


1